In [ ]:
import re

patterns = {
    "ymd_kor": re.compile(r"(?P<y>\d{4})년\s*(?P<m>\d{1,2})월\s*(?P<d>\d{1,2})일"),
    "ymd_dot": re.compile(r"(?P<y>\d{4})\.(?P<m>\d{1,2})\.(?P<d>\d{1,2})"),
    "ymd_dash": re.compile(r"(?P<y>\d{4})-(?P<m>\d{1,2})-(?P<d>\d{1,2})"),
    "ymd_slash": re.compile(r"(?P<y>\d{4})/(?P<m>\d{1,2})/(?P<d>\d{1,2})"),
    "mdy_slash": re.compile(r"(?P<m>\d{1,2})/(?P<d>\d{1,2})/(?P<y>\d{4})"),
    "ymd_short": re.compile(r"(?P<y>\d{2})\.(?P<m>\d{1,2})\.(?P<d>\d{1,2})")
    }

def normalize_date(s: str) -> str | None:
    display_s = '""' if s == "" else s

    if not s:
        print(f"{display_s:<22} → {'None':<12} [빈 값]")
        return None

    for p_name, p_regex in patterns.items():
        match = p_regex.search(s)
        
        if match:
            y = int(match['y'])
            m = int(match['m'])
            d = int(match['d'])

            if y < 100:
                y += 2000

            if not (1 <= m <= 12):
                print(f"{display_s:<22} → {'None':<12} [{p_name} · 유효하지 않은 날짜]")
                return None

            days_in_month = [0, 31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
            if not (1 <= d <= days_in_month[m]):
                print(f"{display_s:<22} → {'None':<12} [{p_name} · 유효하지 않은 날짜]")
                return None

            result = f"{y:04d}-{m:02d}-{d:02d}"
            print(f"{display_s:<22} → {result:<12} [{p_name}]")
            return result

    print(f"{display_s:<22} → {'None':<12} [매치 없음]")
    return None

samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]

for sample in samples:
    normalize_date(sample)

2024.12.24             → 2024-12-24   [ymd_dot]
2024-12-24             → 2024-12-24   [ymd_dash]
2024/12/24             → 2024-12-24   [ymd_slash]
24.12.24               → 2024-12-24   [ymd_short]
2024년 12월 24일          → 2024-12-24   [ymd_kor]
2024년 3월 5일            → 2024-03-05   [ymd_kor]
12/24/2024             → 2024-12-24   [mdy_slash]
2024.12.24 14:30       → 2024-12-24   [ymd_dot]
등록일 : 2024.12.24       → 2024-12-24   [ymd_dot]
2024-13-45             → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음                 → None         [매치 없음]
""                     → None         [빈 값]


In [ ]:
import re
import pandas as pd

log_content = """203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
"""

with open('access.log', 'w', encoding='utf-8') as f:
    f.write(log_content.strip())

pattern = re.compile(
    r'^(?P<ip>\S+)\s+\S+\s+\S+\s+'
    r'\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)\s+HTTP/[0-9.]+"\s+'
    r'(?P<status>\d{3})\s+(?P<bytes>\d+|-)\s+'
    r'"[^"]*"\s+"(?P<user_agent>[^"]+)"'
)

parsed_data = []
skipped = 0

with open('access.log', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
            
        m = pattern.match(line)
        if m:
            parsed_data.append(m.groupdict())
        else:
            skipped += 1

print(f"파싱 {len(parsed_data)}줄 · 건너뜀 {skipped}줄\n")

df = pd.DataFrame(parsed_data)

if len(df):
    df['status'] = df['status'].astype(int)
    
    print("── 상태코드별 요청 수 ──")
    print("status")
    print(df["status"].value_counts().sort_index().to_string())
    print()

    print("── 4xx·5xx 발생 경로 상위 5 ──")
    err_df = df[df['status'] >= 400]
    print("path")
    if not err_df.empty:
        print(err_df['path'].value_counts().head(5).to_string())
    print()

    print("── 봇 의심 User-Agent ──")
    BOT = re.compile(r"bot|crawler|spider|python-requests", re.IGNORECASE)
    df['is_bot'] = df['user_agent'].str.contains(BOT, na=False)

    bot_df = df[df['is_bot']]
    print("user_agent")
    if not bot_df.empty:
        print(bot_df['user_agent'].value_counts().to_string())

    bot_ratio = df['is_bot'].mean() * 100
    print(f"  봇 요청 비율 {bot_ratio:.1f}%")

    df.drop(columns=['is_bot']).to_csv('access_report.csv', index=False, encoding='utf-8-sig')
else:
    print("파싱된 데이터가 없어 집계를 수행할 수 없습니다.")

파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
path
/api/search     1
/detail/9981    1

── 봇 의심 User-Agent ──
user_agent
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%
